# Notebook 04: Modelling and Evaluation

## Objectives
- Build a full ML pipeline: feature engineering + classification
- Handle class imbalance using SMOTE oversampling
- Tune hyperparameters via GridSearchCV
- Evaluate the best model using confusion matrix and classification report
- Assess whether the business requirement (≥75% recall on Churn class) is met
- Save the fitted pipeline for use in the Streamlit dashboard

## Inputs
- `outputs/datasets/collection/telco_churn_cleaned.csv`

## Outputs
- `outputs/ml_pipeline/predict_churn/v1/clf_pipeline.pkl`
- `outputs/ml_pipeline/predict_churn/v1/confusion_matrix.png`
- `outputs/ml_pipeline/predict_churn/v1/classification_report.csv`

---

## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

os.chdir('..') if os.path.basename(os.getcwd()) == 'jupyter_notebooks' else None  # adjust if running locally

PIPELINE_DIR = 'outputs/ml_pipeline/predict_churn/v1'
os.makedirs(PIPELINE_DIR, exist_ok=True)

print("Working directory:", os.getcwd())

## 2. Load Cleaned Dataset

In [ ]:
df = pd.read_csv('outputs/datasets/collection/telco_churn_cleaned.csv')
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)

## 3. Define Features and Target

In [ ]:
TARGET = 'Churn'
X = df.drop(columns=[TARGET])
y = df[TARGET]

CATEGORICAL_FEATURES = X.select_dtypes(include='object').columns.tolist()
NUMERIC_FEATURES = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical features:", CATEGORICAL_FEATURES)
print("Numeric features:", NUMERIC_FEATURES)
print("Target distribution:\n", y.value_counts())

## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

## 5. Define Feature Engineering and ML Pipeline

We use a `ColumnTransformer` to apply:
- `OneHotEncoder` to categorical features (drop first category to avoid dummy variable trap)
- `StandardScaler` to numeric features

SMOTE is applied within the pipeline using `imblearn.pipeline` to prevent data leakage — oversampling only happens on the training fold.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), CATEGORICAL_FEATURES),
    ('num', StandardScaler(), NUMERIC_FEATURES),
])

clf_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42)),
])

print("Pipeline defined successfully.")
print(clf_pipeline)

## 6. Hyperparameter Search with GridSearchCV

We optimise for **recall** on the Churn class (label = 1) since the business requirement prioritises catching churners over minimising false alarms.

In [ ]:
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
    'classifier__class_weight': ['balanced', None],
}

grid_search = GridSearchCV(
    clf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("\nBest parameters:", grid_search.best_params_)
print("Best CV recall score: {:.4f}".format(grid_search.best_score_))

## 7. Evaluate Best Model on Test Set

In [ ]:
best_pipeline = grid_search.best_estimator_

y_pred_train = best_pipeline.predict(X_train)
y_pred_test = best_pipeline.predict(X_test)

label_map = ['No Churn', 'Churn']

print("=" * 55)
print("TRAIN SET — Classification Report")
print("=" * 55)
print(classification_report(y_train, y_pred_train, target_names=label_map))

print("=" * 55)
print("TEST SET — Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred_test, target_names=label_map))

## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_map)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig(os.path.join(PIPELINE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print("Saved: confusion_matrix.png")

## 9. Save Classification Report as CSV

In [ ]:
report_dict = classification_report(
    y_test, y_pred_test, target_names=label_map, output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose().round(2)
report_df.to_csv(os.path.join(PIPELINE_DIR, 'classification_report.csv'))
print(report_df)
print("\nSaved: classification_report.csv")

## 10. Business Requirement Assessment

In [ ]:
churn_recall = report_dict['Churn']['recall']
threshold = 0.75

print(f"Churn class recall on test set: {churn_recall:.4f} ({churn_recall*100:.1f}%)")
print(f"Business requirement threshold: {threshold*100:.0f}%")

if churn_recall >= threshold:
    print("\n✅ BUSINESS REQUIREMENT MET: The model achieved ≥75% recall on the Churn class.")
else:
    print("\n❌ BUSINESS REQUIREMENT NOT MET: Recall is below the 75% target.")
    print("   Consider adjusting the decision threshold or trying a different algorithm.")

## 11. Save Pipeline

In [ ]:
pipeline_path = os.path.join(PIPELINE_DIR, 'clf_pipeline.pkl')
joblib.dump(best_pipeline, pipeline_path)
print(f"Pipeline saved to: {pipeline_path}")

---
## Conclusions
- A full ML pipeline was built incorporating OneHotEncoding, StandardScaling, SMOTE oversampling, and a RandomForestClassifier.
- Hyperparameters were tuned using 5-fold GridSearchCV optimising for recall on the Churn class.
- The fitted pipeline was evaluated on a held-out test set (20% of data).
- The business requirement assessment result is printed above — see cell 10 output.
- The pipeline was saved as `clf_pipeline.pkl` for use by the Streamlit Churn Predictor page.